In [22]:
import os
import glob
import json
import gemmi

In [59]:
# ==========================================
# CONFIGURAÇÕES DO PIPELINE
# ==========================================
DIRETORIO_ENTRADA = "../cif_files" # Ajuste se a pasta dos .cif tiver outro nome
TRAIN_PATH = "../mhc_data/train_templated.json"
ARQUIVO_MANIFESTO_BOLTZ = "../rcsb_processed_targets/manifest.json"
VAL_PATH = "../mhc_data/val_templated.json"
ARQUIVO_LOG = "./log_descarte.txt"
TRAIN_IDS = "../mhc_data/train_templated.txt"
VAL_IDS = "../mhc_data/val_templated.txt"

RAIO_CONTATO_ANGSTROMS = 4.0
MINIMO_CONTATOS_VALIDOS = 10

In [ ]:
# ==========================================
# ESTRUTURAS DE DADOS
# ==========================================

def contar_residuos_validos(cadeia):
    """Conta aminoácidos reais, ignorando moléculas de água e ligantes soltos."""
    return sum(1 for residuo in cadeia if not residuo.is_water())

In [25]:
with open(TRAIN_IDS, 'r') as f:
    train_ids = [line.strip() for line in f]

with open(VAL_IDS, 'r') as f:
    val_ids = [line.strip() for line in f]

In [26]:
train_val = train_ids + val_ids

In [27]:
def obter_mapa_auth_para_label(caminho_cif):
    """
    Lê o arquivo CIF e cria um dicionário traduzindo Auth ID para Label ID.
    Exemplo de retorno: {'A': 'B', 'C': 'A', 'B': 'C'}
    """
    doc = gemmi.cif.read(caminho_cif)
    bloco = doc.sole_block()
    
    # Puxa as duas colunas da tabela principal de átomos
    auth_ids = bloco.find_loop('_atom_site.auth_asym_id')
    label_ids = bloco.find_loop('_atom_site.label_asym_id')
    
    mapa = {}
    # Itera sobre os átomos. Zip é super rápido no Python.
    for auth, label in zip(auth_ids, label_ids):
        if auth not in mapa:
            mapa[auth] = label
            
    return mapa

In [53]:
def mapear_copias_identicas(modelo):
    """Agrupa Auth IDs de cadeias que possuem exatamente a mesma sequência."""
    mapa_seq = {}
    for cadeia in modelo:
        # Monta a sequência da cadeia ignorando água
        seq = "".join([residuo.name for residuo in cadeia if not residuo.is_water()])
        if not seq: continue
        
        if seq not in mapa_seq:
            mapa_seq[seq] = []
        mapa_seq[seq].append(cadeia.name) # Salva o Auth ID
        
    equivalencias = {}
    for copias in mapa_seq.values():
        for cadeia in copias:
            equivalencias[cadeia] = copias
    return equivalencias # Ex: Retorna {'D': ['A', 'D'], 'F': ['C', 'F']}

In [43]:
print("Carregando manifesto original do Boltz para validação cruzada...")
with open(ARQUIVO_MANIFESTO_BOLTZ, 'r') as f:
    dados_boltz = json.load(f)

# Cria um mapa indexado por PDB_ID contendo a lista de cadeias que o Boltz possui
cadeias_disponiveis_boltz = {}
for item in dados_boltz:
    pdb_id = item['id']
    # Coleta todas as letras de cadeias registradas no bloco do Boltz
    letras_boltz = [c['chain_name'] for c in item.get('chains', [])]
    cadeias_disponiveis_boltz[pdb_id] = letras_boltz

Carregando manifesto original do Boltz para validação cruzada...


In [40]:
"A" + "1" in cadeias_disponiveis_boltz["1A1M"]

True

In [ ]:
# ==========================================
# MOTOR PRINCIPAL (BATCH PROCESSING)
# ==========================================

ids=val_ids
# Busca todos os arquivos .cif no diretório
arquivos_cif = glob.glob(os.path.join(DIRETORIO_ENTRADA, "*.cif"))
total_arquivos = len(ids)
logs_descarte = []
manifesto_final = []

print(f"Iniciando processamento de {total_arquivos} estruturas...")

for indice, pdb_id in enumerate(ids, 1):
    # Caminho do arquivo .cif baseado no ID do PDB
    caminho_arquivo = os.path.join(DIRETORIO_ENTRADA, f"{pdb_id}.cif")
    
    # Log de progresso a cada 100 arquivos para acompanhamento no terminal
    if indice % 100 == 0:
        print(f"Progresso: {indice}/{total_arquivos} arquivos processados...")

    try:
        # 1. I/O e Parse
        estrutura = gemmi.read_structure(caminho_arquivo)
        estrutura.setup_entities() # Organiza metadados internos
        modelo = estrutura[0]      # Regra de Negócio: Sempre usar o modelo 0
        
        mapa_traducao = obter_mapa_auth_para_label(caminho_arquivo)
        equivalencias = mapear_copias_identicas(modelo)
        
        candidatos_mhc = []
        candidatos_pep = []

        # 2. Filtro Heurístico Primário (Tamanho)
        for cadeia in modelo:
            # tamanho = contar_residuos_validos(cadeia)
            
            candidatos_mhc.append(cadeia)
            candidatos_pep.append(cadeia)
            # if tamanho > 100:
            #     candidatos_mhc.append(cadeia)
            # elif 5 <= tamanho <= 30:
            #     candidatos_pep.append(cadeia)
        
        # if not candidatos_mhc or not candidatos_pep:
        #     logs_descarte.append(f"{pdb_id} - Faltam candidatos viáveis (MHCs: {len(candidatos_mhc)}, Peps: {len(candidatos_pep)})")
        #     continue

        # 3. Cálculo da Interface (Motor Central Gemmi NeighborSearch)
        # Inicializa a Octree para busca O(log N)
        ns = gemmi.NeighborSearch(modelo, estrutura.cell, RAIO_CONTATO_ANGSTROMS)
        ns.populate(include_h=False) # Regra de Negócio: Ignorar hidrogênios

        melhor_par_auth = None
        maximo_contatos = -1

        # Testa todas as combinações MHC x Peptídeo
        for mhc in candidatos_mhc:
            for pep in candidatos_pep:
                contatos_atuais = 0
                
                # Varre os átomos do peptídeo para achar a superfície do MHC
                for residuo in pep:
                    if residuo.is_water(): continue
                    
                    for atomo in residuo:
                        # Busca vizinhos num raio de 4.0 Å
                        vizinhos = ns.find_atoms(atomo.pos, '\0', radius=RAIO_CONTATO_ANGSTROMS)
                        
                        for marca in vizinhos:
                            # Converte o ponteiro C++ de volta para a estrutura Python
                            vizinho_cra = marca.to_cra(modelo)
                            
                            # Se o átomo vizinho pertence à cadeia do MHC que estamos testando, é um hit!
                            if vizinho_cra.chain.name == mhc.name:
                                contatos_atuais += 1
                                
                # Atualiza o pódio se esta combinação for mais forte
                if contatos_atuais > maximo_contatos:
                    maximo_contatos = contatos_atuais
                    melhor_par_auth = (mhc.name, pep.name)

        # 4. Validação Final e Registro
        if maximo_contatos >= MINIMO_CONTATOS_VALIDOS:
            auth_mhc, auth_pep = melhor_par_auth
            
            irmaos_auth_mhc = equivalencias.get(auth_mhc, [auth_mhc])
            irmaos_auth_pep = equivalencias.get(auth_pep, [auth_pep])
            
            # mhc_label_oficial = mapa_traducao.get(auth_mhc, auth_mhc)
            # pep_label_oficial = mapa_traducao.get(auth_pep, auth_pep)

            # Traduz todos os irmãos para os nomes oficiais de máquina (Label IDs)
            mhc_label_oficial = [mapa_traducao.get(c, c) for c in irmaos_auth_mhc]
            pep_label_oficial = [mapa_traducao.get(c, c) for c in irmaos_auth_pep]

            cadeias_do_boltz = cadeias_disponiveis_boltz.get(pdb_id, [])
            print(f"DEBUG: {pdb_id} - Melhor par encontrado: Auth IDs ({auth_mhc}, {auth_pep}) -> Label IDs ({mhc_label_oficial}, {pep_label_oficial}), Cadeias Boltz: {cadeias_do_boltz}")
            # Busca qual dos irmãos MHC o Boltz resolveu manter no dataset
            for cad in mhc_label_oficial:
                if cad in cadeias_do_boltz or f"{cad}1" in cadeias_do_boltz:  # Trata o sufixo "1" (como A1, C1)
                    mhc_aprovado = cad
                    
            # Busca qual dos irmãos Peptídeo o Boltz resolveu manter
            for cad in pep_label_oficial:
                if cad in cadeias_do_boltz or f"{cad}1" in cadeias_do_boltz:  # Trata o sufixo "1" (como A1, C1)
                    pep_aprovado = cad
            # if mhc_label_oficial + "1" in cadeias_do_boltz and pep_label_oficial + "1" in cadeias_do_boltz:
            # # SUCESSO: O par existe fisicamente e as letras batem com o Boltz
            #     manifesto_final.append({
            #         "pdb_id": pdb_id.lower(),
            #         "protein_chains": [mhc_label_oficial],
            #         "peptide_chain": pep_label_oficial,
            #         "interacoes_atomicas": maximo_contatos,
            #         "auth_ids_originais": melhor_par_auth
            #     })
            if mhc_aprovado and pep_aprovado:
                # SUCESSO: O par existe fisicamente e as letras batem com o Boltz
                manifesto_final.append({
                    "pdb_id": pdb_id.lower(),
                    "protein_chains": [mhc_aprovado],
                    "peptide_chain": pep_aprovado,
                    "interacoes_atomicas": maximo_contatos,
                    "auth_ids_originais": melhor_par_auth
                })
            else:
                # BAD ID EVITADO/CONFIRMADO: A interface existe, mas o Boltz não gerou essas cadeias no NPZ
                logs_descarte.append(
                    f"{pdb_id} - Bad ID Real: Interface física achada ({auth_mhc}-{auth_pep} -> {mhc_label_oficial}-{pep_label_oficial}), "
                    f"mas essas cadeias estão ausentes no arquivo do Boltz. Disponíveis no Boltz: {cadeias_do_boltz}"
                )
                print(f"ALERTA: {pdb_id} - Interface física encontrada, mas cadeias não correspondem ao Boltz. Verifique manualmente.")
            
        else:
            logs_descarte.append(f"{pdb_id} - Interface fraca ou inexistente ({maximo_contatos} contatos)")

    except Exception as erro:
        # Tratamento de Exceções para evitar quebra do batch
        logs_descarte.append(f"{pdb_id} - Falha critica no processamento: {str(erro)}")



print(f"\nPipeline Concluído!")
print(f"Proteínas validadas: {len(manifesto_final)}")
print(f"Proteínas descartadas: {len(logs_descarte)}")
print(f"Resultados salvos em '{VAL_PATH}' e '{ARQUIVO_LOG}'.")


Iniciando processamento de 130 estruturas...
DEBUG: 6zkw - Melhor par encontrado: Auth IDs (A, C) -> Label IDs (['A'], ['C']), Cadeias Boltz: ['A1', 'B1', 'C1', 'D1', 'E1']
DEBUG: 6zkx - Melhor par encontrado: Auth IDs (A, C) -> Label IDs (['A'], ['C']), Cadeias Boltz: ['A1', 'F1', 'G1', 'H1', 'I1', 'B1', 'J1', 'K1', 'L1', 'C1', 'D1', 'M1', 'E1']
DEBUG: 6zky - Melhor par encontrado: Auth IDs (A, C) -> Label IDs (['A'], ['C']), Cadeias Boltz: ['A1', 'B1', 'C1', 'D1', 'E1']
DEBUG: 6zkz - Melhor par encontrado: Auth IDs (A, C) -> Label IDs (['A'], ['C']), Cadeias Boltz: ['A1', 'F1', 'B1', 'G1', 'C1', 'D1', 'H1', 'E1', 'I1']
DEBUG: 7bbg - Melhor par encontrado: Auth IDs (A, C) -> Label IDs (['A'], ['C']), Cadeias Boltz: ['A1', 'F1', 'G1', 'H1', 'B1', 'C1', 'D1', 'I1', 'J1', 'E1']
DEBUG: 7bh8 - Melhor par encontrado: Auth IDs (A, P) -> Label IDs (['A'], ['I', 'J']), Cadeias Boltz: ['A1', 'K1', 'L1', 'M1', 'B1', 'N1', 'O1', 'C1', 'P1', 'D1', 'E1', 'Q1', 'R1', 'F1', 'S1', 'T1', 'G1', 'H1', 'U

In [58]:
# 5. Exportação de Dados (Deliverables)
with open(VAL_PATH, 'w') as f:
    json.dump(manifesto_final, f, indent=4)
    
with open(ARQUIVO_LOG, 'w', encoding="UTF-8") as f:
    f.write('\n'.join(logs_descarte))